# Downstream Retrieval, PCST, And LLM Comparison

This notebook uses the main project config to load its downstream database.
In practice that means it reads the configured node table, edge table, and final GCN embeddings.

It is split into three modules:
1. Retrieve a full, unpruned subgraph.
2. Apply PCST to prune that subgraph.
3. Compare two LLMs on the retrieved evidence.

In [92]:
import json
import os
import re
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
try:
    import ollama
    OLLAMA_AVAILABLE = True
except Exception:
    ollama = None
    OLLAMA_AVAILABLE = False
from pcst_fast import pcst_fast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists() and (REPO_ROOT / ".." / "configs").exists():
    REPO_ROOT = (REPO_ROOT / "..").resolve()

def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    if path.is_absolute():
        return path
    return REPO_ROOT / path

def load_env_file(env_path: Path) -> dict:
    loaded = {}
    if not env_path.exists():
        return loaded
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value
        loaded[key] = value
    return loaded

loaded_env = load_env_file(resolve_path(".env"))
if loaded_env:
    print(f"Loaded .env keys: {sorted(loaded_env)}")
else:
    print("No .env file loaded. You can create one from .env.example.")

with open(resolve_path("configs/config.yaml"), "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Load graph data and embeddings generated by the upstream pipeline.
nodes = pd.read_csv(resolve_path(config["paths"]["nodes_csv"]))
edges = pd.read_csv(resolve_path(config["paths"]["edges_csv"]))
node_embeddings = np.load(resolve_path(config["paths"]["final_embeddings"]))
parsed_text_path = resolve_path(config["paths"].get("parsed_text", "data/processed/parsed_text.json"))
with open(parsed_text_path, "r", encoding="utf-8") as f:
    parsed_pages = json.load(f)

nodes = nodes.copy()
nodes["node_id"] = nodes["node_id"].astype(str)
edges = edges.copy()
edges["source"] = edges["source"].astype(str)
edges["target"] = edges["target"].astype(str)

node_ids = nodes["node_id"].tolist()
node_emb_lookup = dict(zip(node_ids, node_embeddings))

embedding_model_name = config["embedding"]["model_name"]
query_encoder = SentenceTransformer(embedding_model_name)
embedding_dim = int(node_embeddings.shape[1])


print(f"Loaded nodes: {len(nodes):,}")
print(f"Loaded edges: {len(edges):,}")
print(f"Loaded parsed pages: {len(parsed_pages):,}")
print(f"Embedding model: {embedding_model_name}")
print(f"Embedding dimension: {embedding_dim}")


Loaded .env keys: ['ANTHROPIC_API_KEY', 'CLAUDE_MODEL', 'GROQ_API_KEY', 'SEED_SELECTOR_MODEL']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded nodes: 974
Loaded edges: 3,873
Loaded parsed pages: 144
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


## Ollama Setup

1. In terminal, pull the models once:
   - `ollama pull llama3.2:1b`
   - `ollama pull llama3.2:3b`
   - `ollama pull qwen2.5:3b`
2. Start Ollama locally (`ollama serve`) if it is not already running.
3. Restart this notebook kernel and run all cells from the top.


In [93]:
if not OLLAMA_AVAILABLE:
    raise ImportError("ollama package is not installed in this kernel. Run: pip install ollama")

OLLAMA_MODEL_A = os.getenv("OLLAMA_MODEL_A", os.getenv("OLLAMA_MODEL", "llama3.2:1b")).strip()
OLLAMA_MODEL_B = os.getenv("OLLAMA_MODEL_B", "qwen2.5:3b").strip()
OLLAMA_MODEL_C = os.getenv("OLLAMA_MODEL_C", "llama3.2:3b").strip()
OLLAMA_MODELS = [OLLAMA_MODEL_A, OLLAMA_MODEL_B, OLLAMA_MODEL_C]

print("Ollama models to compare:")
for model in OLLAMA_MODELS:
    print(f"- {model}")


Ollama models to compare:
- llama3.2:1b
- qwen2.5:3b
- llama3.2:3b


In [94]:
def test_ollama_connection(model_name: str):
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": "Reply with exactly: connected"}],
    )
    text = response.message.content if hasattr(response, "message") else response["message"]["content"]
    print(f"{model_name}: {text}")

for model in OLLAMA_MODELS:
    test_ollama_connection(model)


llama3.2:1b: No
qwen2.5:3b: connected
llama3.2:3b: connected


## Shared Helpers

In [95]:
def build_graph(edges_df, node_ids=None):
    """Create a simple undirected NetworkX graph from edge rows.

    Parameters
    ----------
    edges_df:
        DataFrame with `source` and `target` columns.
    node_ids:
        Optional node ids to add even if they are isolated.
    """
    graph = nx.Graph()

    if node_ids is not None:
        for node_id in node_ids:
            graph.add_node(node_id)

    for row in edges_df.itertuples(index=False):
        if row.source != row.target:
            graph.add_edge(row.source, row.target)

    return graph

def get_text_rows(node_ids, nodes_df=nodes):
    """Return article / paragraph rows in the same order as the input ids."""
    requested = [str(node_id) for node_id in node_ids]
    order = {node_id: idx for idx, node_id in enumerate(requested)}

    matches = nodes_df[nodes_df["node_id"].isin(requested)].copy()
    matches["request_order"] = matches["node_id"].map(order)

    return matches.sort_values("request_order")[["node_id", "title", "type", "text"]].reset_index(drop=True)

def plot_subgraph(graph, title, nodes_df=nodes, score_lookup=None):
    """Visualise any retrieved graph with Plotly.

    If `score_lookup` is provided, similarity / prize values are shown in the hover text
    and the node size is increased by the prize value.
    """
    type_lookup = nodes_df[["node_id", "type"]].drop_duplicates("node_id").set_index("node_id")["type"].astype(str).to_dict()

    try:
        layout = nx.spring_layout(graph, seed=7)
    except TypeError:
        layout = nx.spring_layout(graph)

    edge_x = []
    edge_y = []
    for source, target in graph.edges():
        x0, y0 = layout[source]
        x1, y1 = layout[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line={"width": 1, "color": "#94a3b8"},
        hoverinfo="none",
    )

    palette = {
        "article": "#1f77b4",
        "paragraph": "#ff7f0e",
        "annex": "#2ca02c",
        "recital": "#8c564b",
    }

    node_x = []
    node_y = []
    node_text = []
    node_label = []
    node_color = []
    node_size = []

    for node_id in graph.nodes():
        x, y = layout[node_id]
        node_type = type_lookup.get(node_id, "unknown")
        score = {} if score_lookup is None else score_lookup.get(node_id, {})
        similarity = score.get("similarity", np.nan)
        prize = score.get("prize", 0.0)

        node_x.append(x)
        node_y.append(y)
        node_label.append(node_id)
        node_color.append(palette.get(node_type.lower(), "#7f7f7f"))
        node_size.append(16 + (6 * prize))
        node_text.append(
            f"node_id={node_id}<br>type={node_type}<br>degree={graph.degree(node_id)}"
            f"<br>similarity={similarity:.4f}<br>prize={prize:.2f}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_label,
        textposition="top center",
        hoverinfo="text",
        hovertext=node_text,
        marker={
            "size": node_size,
            "color": node_color,
            "line": {"width": 1, "color": "#ffffff"},
            "opacity": 0.9,
        },
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=title,
        template="plotly_white",
        showlegend=False,
        xaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
        yaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
        margin={"l": 20, "r": 20, "t": 60, "b": 20},
        height=650,
    )

    return fig

# Build the full graph once. Both modules reuse it.
full_graph = build_graph(edges, node_ids=node_ids)

## Module 1: Retrieve An Unpruned Subgraph

In [96]:
def encode_query(query):
    """Encode the raw query with the embedding model from config.

    The returned query vector is used for both retrieval and PCST scoring.
    Its width must match the saved node embedding width.
    """
    query_emb = query_encoder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    if query_emb.shape[0] != embedding_dim:
        raise ValueError(
            f"Query embedding dimension {query_emb.shape[0]} does not match node embedding dimension {embedding_dim}."
        )
    return query_emb


def _call_selector_model(prompt, selector_model, system_prompt):
    if not OLLAMA_AVAILABLE:
        raise ImportError("ollama package is not installed in this kernel. Run: pip install ollama")
    response = ollama.chat(
        model=selector_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )
    text = response.message.content if hasattr(response, "message") else response["message"]["content"]
    return text, selector_model


def build_page_selection_prompt(query, page_df, target_pages):
    blocks = []
    for row in page_df.itertuples(index=False):
        blocks.append(
            f"page_number: {row.page_number}\n"
            f"preview: {row.preview}"
        )
    pages_text = "\n\n---\n\n".join(blocks)
    return f"""Given the user question and EU AI Act page previews, select the {target_pages} most relevant pages.

Question:
{query}

Page previews:
{pages_text}

Return JSON only in this format:
{{"selected_pages": [12, 34, 55], "reason": "short explanation"}}
"""


def parse_selected_pages(raw_text, valid_pages, target_pages):
    text = raw_text.strip()
    json_candidates = [text]
    fenced = re.findall(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL)
    json_candidates.extend(fenced)
    parsed_pages = []
    for candidate in json_candidates:
        try:
            payload = json.loads(candidate)
        except Exception:
            continue
        if isinstance(payload, dict):
            parsed_pages = payload.get("selected_pages", [])
        elif isinstance(payload, list):
            parsed_pages = payload
        if parsed_pages:
            break
    out = []
    for p in parsed_pages:
        try:
            p_int = int(p)
        except Exception:
            continue
        if p_int in valid_pages and p_int not in out:
            out.append(p_int)
    if len(out) < target_pages:
        for p in valid_pages:
            if p not in out:
                out.append(p)
            if len(out) >= target_pages:
                break
    return out[:target_pages]


def build_node_selection_prompt(query, node_candidate_df, k):
    blocks = []
    for row in node_candidate_df.itertuples(index=False):
        blocks.append(
            f"node_id: {row.node_id}\n"
            f"type: {row.type}\n"
            f"title: {row.title}\n"
            f"page: {row.page}\n"
            f"text_preview: {row.preview}"
        )
    candidates_text = "\n\n---\n\n".join(blocks)
    return f"""Select the {k} best seed nodes for graph retrieval, using only these node candidates derived from parsed_text pages.

Question:
{query}

Node candidates:
{candidates_text}

Return JSON only in this format:
{{"selected_node_ids": ["node_id_1", "node_id_2"], "reason": "short explanation"}}
"""


def parse_selected_node_ids(raw_text, valid_node_ids, k):
    text = raw_text.strip()
    valid_node_ids = [str(node_id) for node_id in valid_node_ids]

    json_candidates = [text]
    fenced = re.findall(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL)
    json_candidates.extend(fenced)

    parsed_node_ids = []
    for candidate in json_candidates:
        try:
            payload = json.loads(candidate)
        except Exception:
            continue
        if isinstance(payload, dict):
            parsed_node_ids = payload.get("selected_node_ids", [])
        elif isinstance(payload, list):
            parsed_node_ids = payload
        if parsed_node_ids:
            break

    selected = []
    for node_id in parsed_node_ids:
        node_id = str(node_id)
        if node_id in valid_node_ids and node_id not in selected:
            selected.append(node_id)

    if len(selected) < k:
        for node_id in valid_node_ids:
            if node_id not in selected:
                selected.append(node_id)
            if len(selected) >= k:
                break

    return selected[:k]


def retrieve_seed_nodes(query, k=2, model_id=None, page_select_k=8):
    """LLM-first seed selection from parsed_text.json (no cosine shortlist)."""
    selector_model = model_id or os.getenv("SEED_SELECTOR_MODEL", os.getenv("OLLAMA_MODEL", "llama3.2:1b"))

    page_rows = []
    for page in parsed_pages:
        page_num = int(page.get("page_number", -1))
        text = str(page.get("text", "")).strip()
        if page_num <= 0 or not text:
            continue
        page_rows.append({
            "page_number": page_num,
            "preview": text[:1400].replace("\n", " "),
        })
    page_df = pd.DataFrame(page_rows).sort_values("page_number").reset_index(drop=True)
    valid_pages = page_df["page_number"].astype(int).tolist()
    if page_df.empty:
        return [], pd.DataFrame(), "", selector_model

    page_prompt = build_page_selection_prompt(query, page_df, target_pages=page_select_k)
    page_raw, used_model = _call_selector_model(
        prompt=page_prompt,
        selector_model=selector_model,
        system_prompt="You are selecting relevant legal pages. Return valid JSON only.",
    )
    selected_pages = parse_selected_pages(page_raw, valid_pages=valid_pages, target_pages=page_select_k)

    node_candidates = nodes[nodes["page"].isin(selected_pages)].copy()
    if node_candidates.empty:
        node_candidates = nodes.copy()
    node_candidates = node_candidates.copy()
    node_candidates["node_id"] = node_candidates["node_id"].astype(str)
    node_candidates["title"] = node_candidates["title"].fillna("").astype(str)
    node_candidates["type"] = node_candidates["type"].fillna("").astype(str)
    node_candidates["text"] = node_candidates["text"].fillna("").astype(str)
    node_candidates["preview"] = node_candidates["text"].str[:700].str.replace("\n", " ", regex=False)
    candidate_df = node_candidates[["node_id", "type", "title", "page", "preview"]].reset_index(drop=True)
    valid_node_ids = candidate_df["node_id"].tolist()

    node_prompt = build_node_selection_prompt(query, candidate_df, k=k)
    node_raw, used_model = _call_selector_model(
        prompt=node_prompt,
        selector_model=used_model,
        system_prompt="You are selecting legal seed nodes. Return valid JSON only.",
    )
    selected_nodes = parse_selected_node_ids(node_raw, valid_node_ids=valid_node_ids, k=k)
    return selected_nodes, candidate_df, node_raw, used_model

def expand_k_hops(graph, seeds, k=2):
    """Return the full k-hop subgraph around the seed nodes."""
    valid_seeds = [seed for seed in seeds if seed in graph]
    nodes_in_scope = set(valid_seeds)
    frontier = set(valid_seeds)

    for _ in range(k):
        next_frontier = set()
        for node_id in frontier:
            next_frontier.update(graph.neighbors(node_id))
        frontier = next_frontier
        nodes_in_scope.update(next_frontier)

    return graph.subgraph(nodes_in_scope).copy()

def retrieve_unpruned_subgraph(query, num_retrieved_seeds=5, k_hops=2, selector_model=None):
    """Module 1.

    Input:
    - raw text query
    - number of LLM-selected seed nodes
    - number of hop expansions

    Output:
    - the full retrieved graph before pruning
    - the corresponding article / paragraph rows
    - a Plotly visualisation
    """
    query_emb = encode_query(query)
    seed_nodes, candidate_df, raw_seed_selection, used_model = retrieve_seed_nodes(
        query,
        k=num_retrieved_seeds,
        model_id=selector_model,
    )
    unpruned_graph = expand_k_hops(full_graph, seed_nodes, k=k_hops)

    retrieved_node_ids = list(unpruned_graph.nodes())
    retrieved_edges_df = pd.DataFrame(list(unpruned_graph.edges()), columns=["source", "target"])
    retrieved_text_df = get_text_rows(retrieved_node_ids)
    retrieved_figure = plot_subgraph(
        unpruned_graph,
        title=f"Unpruned retrieved subgraph for query: {query}",
    )

    return {
        "query": query,
        "query_emb": query_emb,
        "selector_backend": "ollama",
        "selector_model": used_model,
        "seed_nodes": seed_nodes,
        "seed_candidate_df": candidate_df,
        "raw_seed_selection": raw_seed_selection,
        "subgraph": unpruned_graph,
        "subgraph_node_ids": retrieved_node_ids,
        "subgraph_edges_df": retrieved_edges_df,
        "text_df": retrieved_text_df,
        "figure": retrieved_figure,
    }


## Module 2: Apply PCST To The Retrieved Subgraph

In [97]:
def assign_rank_prizes(node_ids, query_emb, node_emb_lookup, top_k=10):
    """Assign node prizes from cosine similarity ranks.

    Only the top-k most similar nodes receive non-zero prizes.
    Higher similarity means a larger prize.
    """
    embs = np.vstack([node_emb_lookup[node_id] for node_id in node_ids])
    similarities = cosine_similarity([query_emb], embs)[0]

    prizes = np.zeros(len(node_ids), dtype=float)
    ranked_idx = np.argsort(similarities)[::-1][:min(top_k, len(similarities))]
    for rank, idx in enumerate(ranked_idx):
        prizes[idx] = top_k - rank

    return prizes, similarities

def graph_to_pcst_input(graph, prizes, edge_cost=1.0):
    """Convert a NetworkX graph into the array representation required by pcst_fast."""
    node_list = list(graph.nodes())
    node_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}

    edge_pairs = []
    edge_costs = []
    for source, target in graph.edges():
        edge_pairs.append((node_to_idx[source], node_to_idx[target]))
        edge_costs.append(edge_cost)

    return (
        node_list,
        np.asarray(edge_pairs, dtype=np.int32),
        np.asarray(prizes, dtype=np.float64),
        np.asarray(edge_costs, dtype=np.float64),
    )

def run_node_prize_pcst(graph, prizes, edge_cost=1.0, pruning="strong"):
    """Run PCST and map the selected vertex / edge indices back to node ids."""
    node_list, edge_array, prize_array, cost_array = graph_to_pcst_input(graph, prizes, edge_cost=edge_cost)
    selected_vertex_idx, selected_edge_idx = pcst_fast(edge_array, prize_array, cost_array, -1, 1, pruning, 0)

    selected_nodes = [node_list[idx] for idx in selected_vertex_idx]
    selected_node_set = set(selected_nodes)
    selected_edges = []
    edge_list = list(graph.edges())
    for edge_idx in selected_edge_idx:
        source, target = edge_list[edge_idx]
        if source in selected_node_set and target in selected_node_set:
            selected_edges.append((source, target))

    return selected_nodes, selected_edges

def prune_retrieved_subgraph(retrieval_result, prize_top_k=10, edge_cost=1.0):
    """Module 2.

    Input:
    - the output of Module 1
    - PCST prize top-k setting
    - constant edge cost

    Output:
    - the pruned graph
    - the corresponding article / paragraph rows
    - a Plotly visualisation
    """
    candidate_graph = retrieval_result["subgraph"]
    candidate_node_ids = retrieval_result["subgraph_node_ids"]

    candidate_node_emb_lookup = {
        node_id: node_emb_lookup[node_id]
        for node_id in candidate_node_ids
    }

    prizes, similarities = assign_rank_prizes(
        node_ids=candidate_node_ids,
        query_emb=retrieval_result["query_emb"],
        node_emb_lookup=candidate_node_emb_lookup,
        top_k=prize_top_k,
    )

    selected_nodes, selected_edges = run_node_prize_pcst(
        candidate_graph,
        prizes,
        edge_cost=edge_cost,
        pruning="strong",
    )

    node_scores_df = pd.DataFrame({
        "node_id": candidate_node_ids,
        "similarity": similarities,
        "prize": prizes,
    }).sort_values(["prize", "similarity"], ascending=False)

    selected_edges_df = pd.DataFrame(selected_edges, columns=["source", "target"])
    pruned_graph = build_graph(selected_edges_df, node_ids=selected_nodes)
    pruned_text_df = get_text_rows(selected_nodes)
    score_lookup = node_scores_df.set_index("node_id").to_dict("index")
    pruned_figure = plot_subgraph(
        pruned_graph,
        title=f"PCST-pruned subgraph for query: {retrieval_result['query']}",
        score_lookup=score_lookup,
    )

    return {
        "query": retrieval_result["query"],
        "selected_nodes": selected_nodes,
        "selected_edges_df": selected_edges_df,
        "node_scores_df": node_scores_df,
        "pruned_graph": pruned_graph,
        "text_df": pruned_text_df,
        "figure": pruned_figure,
    }

## Example: Compare Ollama Seed Selection (`llama3.2:1b` vs `qwen2.5:3b` vs `llama3.2:3b`)


In [108]:
query = "What is AI literacy as described in Article 4 of the AI Act?"

llama1b_retrieval = retrieve_unpruned_subgraph(
    query=query,
    num_retrieved_seeds=5,
    k_hops=1,
    selector_model=OLLAMA_MODELS[0],
)

qwen_retrieval = retrieve_unpruned_subgraph(
    query=query,
    num_retrieved_seeds=5,
    k_hops=1,
    selector_model=OLLAMA_MODELS[1],
)

llama3b_retrieval = retrieve_unpruned_subgraph(
    query=query,
    num_retrieved_seeds=5,
    k_hops=1,
    selector_model=OLLAMA_MODELS[2],
)

pd.DataFrame([
    {
        "model": OLLAMA_MODELS[0],
        "backend": llama1b_retrieval["selector_backend"],
        "selector_model": llama1b_retrieval["selector_model"],
        "seed_nodes": ", ".join(llama1b_retrieval["seed_nodes"]),
        "num_subgraph_nodes": len(llama1b_retrieval["subgraph_node_ids"]),
    },
    {
        "model": OLLAMA_MODELS[1],
        "backend": qwen_retrieval["selector_backend"],
        "selector_model": qwen_retrieval["selector_model"],
        "seed_nodes": ", ".join(qwen_retrieval["seed_nodes"]),
        "num_subgraph_nodes": len(qwen_retrieval["subgraph_node_ids"]),
    },
    {
        "model": OLLAMA_MODELS[2],
        "backend": llama3b_retrieval["selector_backend"],
        "selector_model": llama3b_retrieval["selector_model"],
        "seed_nodes": ", ".join(llama3b_retrieval["seed_nodes"]),
        "num_subgraph_nodes": len(llama3b_retrieval["subgraph_node_ids"]),
    }
])


,model,backend,selector_model,seed_nodes,num_subgraph_nodes
0,llama3.2:1b,ollama,llama3.2:1b,"recital_1, recital_2, recital_3, recital_4, re...",12
1,qwen2.5:3b,ollama,qwen2.5:3b,"article_8, article_8_para_1, recital_1, recita...",14
2,llama3.2:3b,ollama,llama3.2:3b,"article_8, article_8_para_1, article_8_para_2,...",13


In [109]:
for label, result in [
    (OLLAMA_MODELS[0], llama1b_retrieval),
    (OLLAMA_MODELS[1], qwen_retrieval),
    (OLLAMA_MODELS[2], llama3b_retrieval),
]:
    print("=" * 100)
    print(label)
    print("=" * 100)
    candidate_preview = result["seed_candidate_df"][["node_id", "type", "title", "page", "preview"]].copy()
    display(candidate_preview)
    print("Raw seed selection")
    print(result["raw_seed_selection"])
    print()


llama3.2:1b


,node_id,type,title,page,preview
0,recital_1,recital,Recital (1),1,(1) The purpose of this Regulation is to impro...
1,recital_2,recital,Recital (2),1,(2) This Regulation should be applied in accor...
2,recital_3,recital,Recital (3),1,(3) AI systems can be easily deployed in a lar...
3,recital_4,recital,Recital (4),1,(4) Position of the European Parliament of 13 ...
4,recital_5,recital,Recital (5),2,"(5) At the same time, depending on the circums..."
5,recital_6,recital,Recital (6),2,(6) Given the major impact that AI can have on...
6,recital_7,recital,Recital (7),2,(7) In order to ensure a consistent and high l...
7,recital_8,recital,Recital (8),2,(8) A Union legal framework laying down harmon...
8,recital_9,recital,Recital (9),3,(9) Harmonised rules applicable to the placing...
9,recital_10,recital,Recital (10),3,(10) remain unaffected and fully applicable. F...


Raw seed selection
Here is the response in the requested JSON format:

```
{
  "selected_node_ids": ["node_id_1", "node_id_2"],
  "reason": "prohibition of AI practices with unacceptable consequences"
}
```

This JSON object indicates that the selected nodes (node_ids) are those with prohibitions or unacceptable AI practices with significant consequences (reason).

qwen2.5:3b


,node_id,type,title,page,preview
0,recital_1,recital,Recital (1),1,(1) The purpose of this Regulation is to impro...
1,recital_2,recital,Recital (2),1,(2) This Regulation should be applied in accor...
2,recital_3,recital,Recital (3),1,(3) AI systems can be easily deployed in a lar...
3,recital_4,recital,Recital (4),1,(4) Position of the European Parliament of 13 ...
4,recital_5,recital,Recital (5),2,"(5) At the same time, depending on the circums..."
5,recital_6,recital,Recital (6),2,(6) Given the major impact that AI can have on...
6,recital_7,recital,Recital (7),2,(7) In order to ensure a consistent and high l...
7,recital_8,recital,Recital (8),2,(8) A Union legal framework laying down harmon...
8,recital_9,recital,Recital (9),3,(9) Harmonised rules applicable to the placing...
9,recital_10,recital,Recital (10),3,(10) remain unaffected and fully applicable. F...


Raw seed selection
{"selected_node_ids": ["article_8", "article_8_para_1"], "reason": "These articles and paragraphs directly address the compliance requirements for high-risk AI systems, which is central to the regulation's main objective of ensuring safety and efficacy."}

llama3.2:3b


,node_id,type,title,page,preview
0,recital_1,recital,Recital (1),1,(1) The purpose of this Regulation is to impro...
1,recital_2,recital,Recital (2),1,(2) This Regulation should be applied in accor...
2,recital_3,recital,Recital (3),1,(3) AI systems can be easily deployed in a lar...
3,recital_4,recital,Recital (4),1,(4) Position of the European Parliament of 13 ...
4,recital_5,recital,Recital (5),2,"(5) At the same time, depending on the circums..."
5,recital_6,recital,Recital (6),2,(6) Given the major impact that AI can have on...
6,recital_7,recital,Recital (7),2,(7) In order to ensure a consistent and high l...
7,recital_8,recital,Recital (8),2,(8) A Union legal framework laying down harmon...
8,recital_9,recital,Recital (9),3,(9) Harmonised rules applicable to the placing...
9,recital_10,recital,Recital (10),3,(10) remain unaffected and fully applicable. F...


Raw seed selection
{"selected_node_ids": ["article_8", "article_8_para_1", "article_8_para_2"], "reason": "This is the selected output"}



In [100]:
llama1b_pruned = prune_retrieved_subgraph(
    llama1b_retrieval,
    prize_top_k=5,
    edge_cost=1.0,
)

qwen_pruned = prune_retrieved_subgraph(
    qwen_retrieval,
    prize_top_k=5,
    edge_cost=1.0,
)

llama3b_pruned = prune_retrieved_subgraph(
    llama3b_retrieval,
    prize_top_k=5,
    edge_cost=1.0,
)

pd.DataFrame([
    {
        "model": OLLAMA_MODELS[0],
        "seed_nodes": ", ".join(llama1b_retrieval["seed_nodes"]),
        "retrieved_nodes": len(llama1b_retrieval["subgraph_node_ids"]),
        "pcst_nodes": len(llama1b_pruned["selected_nodes"]),
    },
    {
        "model": OLLAMA_MODELS[1],
        "seed_nodes": ", ".join(qwen_retrieval["seed_nodes"]),
        "retrieved_nodes": len(qwen_retrieval["subgraph_node_ids"]),
        "pcst_nodes": len(qwen_pruned["selected_nodes"]),
    },
    {
        "model": OLLAMA_MODELS[2],
        "seed_nodes": ", ".join(llama3b_retrieval["seed_nodes"]),
        "retrieved_nodes": len(llama3b_retrieval["subgraph_node_ids"]),
        "pcst_nodes": len(llama3b_pruned["selected_nodes"]),
    }
])


,model,seed_nodes,retrieved_nodes,pcst_nodes
0,llama3.2:1b,"recital_1, recital_2",4,3
1,qwen2.5:3b,"article_8, article_8_para_1",9,4
2,llama3.2:3b,"recital_20, recital_21",4,3


In [101]:
for label, retrieval_df, pruned_df in [
    (OLLAMA_MODELS[0], llama1b_retrieval["text_df"], llama1b_pruned["text_df"]),
    (OLLAMA_MODELS[1], qwen_retrieval["text_df"], qwen_pruned["text_df"]),
    (OLLAMA_MODELS[2], llama3b_retrieval["text_df"], llama3b_pruned["text_df"]),
]:
    print("=" * 100)
    print(label)
    print("=" * 100)
    print("Retrieved subgraph text rows")
    display(retrieval_df)
    print("PCST-pruned text rows")
    display(pruned_df)


llama3.2:1b
Retrieved subgraph text rows


,node_id,title,type,text
0,recital_1,Recital (1),recital,(1) The purpose of this Regulation is to impro...
1,recital_2,Recital (2),recital,(2) This Regulation should be applied in accor...
2,definition_11,Definition: putting into service,definition,(11) ‘putting into service’ means the supply o...
3,definition_9,Definition: placing on the market,definition,(9) ‘placing on the market’ means the first ma...


PCST-pruned text rows


,node_id,title,type,text
0,recital_1,Recital (1),recital,(1) The purpose of this Regulation is to impro...
1,definition_11,Definition: putting into service,definition,(11) ‘putting into service’ means the supply o...
2,definition_9,Definition: placing on the market,definition,(9) ‘placing on the market’ means the first ma...


qwen2.5:3b
Retrieved subgraph text rows


...

,node_id,title,type,text
0,recital_38,Recital (38),recital,(38) The use of AI systems for real-time remot...
1,article_8,Article 8: Compliance with the requirements,article,Article 8\nCompliance with the requirements\n1...
2,annex_I,Annex I: List of Union harmonisation legislation,annex,ANNEX I\nList of Union harmonisation legislati...
3,article_8_para_2,"Article 8, Paragraph 2",paragraph,"2. Where a product contains an AI system, to w..."


llama3.2:3b
Retrieved subgraph text rows


,node_id,title,type,text
0,recital_21,Recital (21),recital,(21) In order to ensure a level playing field ...
1,definition_1,Definition: AI system,definition,(1) ‘AI system’ means a machine-based system t...
2,recital_20,Recital (20),recital,(20) In order to obtain the greatest benefits ...
3,definition_56,Definition: AI literacy,definition,"(56) ‘AI literacy’ means skills, knowledge and..."


PCST-pruned text rows


,node_id,title,type,text
0,definition_1,Definition: AI system,definition,(1) ‘AI system’ means a machine-based system t...
1,recital_20,Recital (20),recital,(20) In order to obtain the greatest benefits ...
2,definition_56,Definition: AI literacy,definition,"(56) ‘AI literacy’ means skills, knowledge and..."


## Figures

In [102]:
llama1b_retrieval["figure"]


In [103]:
qwen_retrieval["figure"]


In [104]:
llama3b_retrieval["figure"]


In [105]:
llama1b_pruned["figure"]


In [106]:
qwen_pruned["figure"]


In [107]:
llama3b_pruned["figure"]
